# Лабораторна робота №4. Побудова та дослідження наївного байєсівського класифікатора

## Проєкт DocVerify - класифікація типу бізнес-документа та перевірка автентичності

Ця лабораторна робота продовжує Лаб.2 та Лаб.3: використовуємо той самий простір ознак (`data/features_lab2.csv` після препроцесингу — сентинел-заповнення `authenticity`, IQR-capping `width`/`file_size_bytes`, One-Hot кодування `source_dataset`/`authenticity`, стандартизація 6 числових ознак, видалення `mean_brightness`) і те саме розбиття train/test (`test_size=0.2`, `random_state=42`, `stratify=y`), щоб результати цієї лабораторної були повністю порівнянні з $k$-NN із Лаб.3.

**План роботи:**

1. **Частина 1.** Побудова та оцінка класифікатора GaussianNB на даних командного проєкту DocVerify (оцінка Accuracy, F1-macro, часу навчання та передбачення).
2. **Частина 2.** Дослідження MultinomialNB для класифікації текстових даних: вивчення впливу параметра згладжування Лапласа ($\alpha = 1.0, 0.0, 0.01$) на якість моделі та розв'язання проблеми нульової ймовірності.
3. **Частина 3.** Порівняльний аналіз моделей: порівняння Наївного Байєса та $k$-NN за точністю та обчислювальною складністю («ліниве» vs «жадібне» навчання).
4. **Підсумкові висновки.**

In [5]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.datasets import fetch_20newsgroups

# Налаштування відображення
pd.set_option('display.max_columns', None)

## 1. Завантаження даних та відтворення препроцесингу з Лаб №2 / Лаб №3

Завантажуємо датасет `data/features_lab2.csv` і застосовуємо повністю ідентичний пайплайн підготовки даних, щоб зберегти об'єктивність порівняння.

In [6]:
FEATURES_CSV = "data/features_lab2.csv"
TARGET = "document_type"
AUTH_COLUMN = "authenticity"
RANDOM_STATE = 42
TEST_SIZE = 0.2

# 1. Завантаження даних
try:
    df = pd.read_csv(FEATURES_CSV)
    print(f"Датасет успішно завантажено. Розмірність: {df.shape}")
except FileNotFoundError:
    print("Увага: Файл не знайдено, використовується генерація синтетичних даних для демонстрації.")
    data = {
        'width': np.random.randint(500, 1500, 1000),
        'height': np.random.randint(500, 1500, 1000),
        'file_size_bytes': np.random.randint(10000, 500000, 1000),
        'aspect_ratio': np.random.uniform(0.5, 1.5, 1000),
        'std_brightness': np.random.uniform(10, 80, 1000),
        'dark_pixel_ratio': np.random.uniform(0.1, 0.9, 1000),
        'source_dataset': np.random.choice(['RVL-CDIP', 'CORD', 'Kaggle', 'MIDV-500'], 1000),
        'authenticity': np.random.choice(['genuine', np.nan], 1000, p=[0.1, 0.9]),
        'document_type': np.random.choice([f'type_{i}' for i in range(20)], 1000)
    }
    df = pd.DataFrame(data)

# 2. Обробка пропущених значень
df[AUTH_COLUMN] = df[AUTH_COLUMN].fillna('not_applicable')

# 3. Обробка викидів (IQR capping)
def iqr_bounds(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ['width', 'file_size_bytes']:
    if col in df.columns:
        lo, hi = iqr_bounds(df[col])
        df[col] = df[col].clip(lower=lo, upper=hi)

if 'mean_brightness' in df.columns:
    df = df.drop(columns=['mean_brightness'])

# 4. Виділення ознак та розбиття train/test
NUMERIC_COLS_SELECTED = ['width', 'height', 'file_size_bytes', 'aspect_ratio', 'std_brightness', 'dark_pixel_ratio']
CATEGORICAL_COLS = ['source_dataset', AUTH_COLUMN]

le_target = LabelEncoder()
y = le_target.fit_transform(df[TARGET])

train_idx, test_idx = train_test_split(
    df.index, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

df_train, df_test = df.loc[train_idx], df.loc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# 5. Масштабування та One-Hot кодування
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.fit(df_train[CATEGORICAL_COLS])
X_train_cat = ohe.transform(df_train[CATEGORICAL_COLS])
X_test_cat = ohe.transform(df_test[CATEGORICAL_COLS])

scaler = StandardScaler()
scaler.fit(df_train[NUMERIC_COLS_SELECTED])
X_train_num = scaler.transform(df_train[NUMERIC_COLS_SELECTED])
X_test_num = scaler.transform(df_test[NUMERIC_COLS_SELECTED])

X_train_processed = np.hstack([X_train_num, X_train_cat])
X_test_processed = np.hstack([X_test_num, X_test_cat])

print(f"Розмірність обробленої навчальної вибірки: {X_train_processed.shape}")
print(f"Розмірність обробленої тестової вибірки: {X_test_processed.shape}")

Датасет успішно завантажено. Розмірність: (3682, 12)
Розмірність обробленої навчальної вибірки: (2945, 13)
Розмірність обробленої тестової вибірки: (737, 13)


## 2. Частина 1: Навчання та оцінка класифікатора GaussianNB

Будуємо гаусівський наївний байєсівський класифікатор `GaussianNB`, вимірюємо час навчання (`fit`) та час передбачення (`predict`), а також обчислюємо основні метрики якості.

In [7]:
gnb = GaussianNB()

# Замір часу навчання
t0 = time.perf_counter()
gnb.fit(X_train_processed, y_train)
gnb_fit_time = time.perf_counter() - t0

# Замір часу передбачення
t0 = time.perf_counter()
y_pred_gnb = gnb.predict(X_test_processed)
gnb_predict_time = time.perf_counter() - t0

# Метрики
gnb_acc = accuracy_score(y_test, y_pred_gnb)
gnb_prec = precision_score(y_test, y_pred_gnb, average='macro', zero_division=0)
gnb_rec = recall_score(y_test, y_pred_gnb, average='macro', zero_division=0)
gnb_f1 = f1_score(y_test, y_pred_gnb, average='macro', zero_division=0)

print(f"--- Результати GaussianNB ---")
print(f"Accuracy:        {gnb_acc:.4f}")
print(f"Precision-macro: {gnb_prec:.4f}")
print(f"Recall-macro:    {gnb_rec:.4f}")
print(f"F1-macro:        {gnb_f1:.4f}")
print(f"Час навчання:    {gnb_fit_time:.6f} сек")
print(f"Час передбачення:{gnb_predict_time:.6f} сек")

--- Результати GaussianNB ---
Accuracy:        0.3297
Precision-macro: 0.3505
Recall-macro:    0.3579
F1-macro:        0.3053
Час навчання:    0.003979 сек
Час передбачення:0.001834 сек


## 3. Частина 2: Дослідження MultinomialNB та згладжування Лапласа ($\alpha$)

Для дослідження мультиноміального наївного Байєса та впливу параметра $\alpha$ використовуємо еталонний датасет текстової класифікації `20newsgroups`. Порівнюємо режими:
- $\alpha = 1.0$ (Класичне згладжування Лапласа)
- $\alpha = 0.0$ (Без згладжування — демонстрація проблеми нульової ймовірності)
- $\alpha = 0.01$ (Слабке згладжування ЛіДстона)

In [8]:
# Завантаження текстових даних
newsgroups = fetch_20newsgroups(
    subset='all', 
    categories=['alt.atheism', 'comp.graphics', 'sci.med', 'talk.religion.misc'], 
    random_state=RANDOM_STATE
)

X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    newsgroups.data, newsgroups.target, test_size=0.2, random_state=RANDOM_STATE, stratify=newsgroups.target
)

# Векторизація тексту (Bag-of-Words)
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_text_train)
X_test_vec = vectorizer.transform(X_text_test)

alpha_values = [1.0, 0.0, 0.01]
mnb_results = {}

for alpha in alpha_values:
    mnb = MultinomialNB(alpha=alpha)
    
    t0 = time.perf_counter()
    mnb.fit(X_train_vec, y_text_train)
    fit_t = time.perf_counter() - t0
    
    t0 = time.perf_counter()
    y_pred_mnb = mnb.predict(X_test_vec)
    pred_t = time.perf_counter() - t0
    
    acc = accuracy_score(y_text_test, y_pred_mnb)
    f1 = f1_score(y_text_test, y_pred_mnb, average='macro', zero_division=0)
    
    mnb_results[alpha] = {
        'accuracy': acc, 'f1_macro': f1, 
        'fit_time': fit_t, 'predict_time': pred_t
    }

# Відображення таблиці результатів для MultinomialNB
mnb_df = pd.DataFrame.from_dict(mnb_results, orient='index')
mnb_df.index.name = 'Alpha'
mnb_df.reset_index(inplace=True)
mnb_df

c:\Users\Mary\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\naive_bayes.py:930: RuntimeWarning: divide by zero encountered in log
  self.feature_log_prob_ = np.log(smoothed_fc) - np.log(


,Alpha,accuracy,f1_macro,fit_time,predict_time
0,1.00,0.945428,0.938652,0.007852,0.001748
1,0.00,0.466077,0.460365,0.009849,0.001313
2,0.01,0.948378,0.941933,0.006370,0.001242


## 4. Частина 3: Порівняльний аналіз Naive Bayes та $k$-NN

Навчаємо модель $k$-NN з оптимальним $k=21$ (з Лабораторної роботи №3) на тих самих даних для чесного порівняння за точністю та обчислювальною складністю.

In [9]:
# Навчання k-NN (k=21)
knn = KNeighborsClassifier(n_neighbors=21, metric='euclidean', weights='uniform')

t0 = time.perf_counter()
knn.fit(X_train_processed, y_train)
knn_fit_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_knn = knn.predict(X_test_processed)
knn_predict_time = time.perf_counter() - t0

knn_acc = accuracy_score(y_test, y_pred_knn)
knn_f1 = f1_score(y_test, y_pred_knn, average='macro', zero_division=0)

# Побудова зведеної порівняльної таблиці
comparison_df = pd.DataFrame([
    {
        'Модель / Метод': 'k-NN (k=21) [DocVerify]',
        'Тип навчання': 'Lazy (Ліниве)',
        'Accuracy': round(knn_acc, 4),
        'F1-macro': round(knn_f1, 4),
        'Час навчання (с)': round(knn_fit_time, 6),
        'Час передбачення (с)': round(knn_predict_time, 6)
    },
    {
        'Модель / Метод': 'GaussianNB [DocVerify]',
        'Тип навчання': 'Eager (Жадібне)',
        'Accuracy': round(gnb_acc, 4),
        'F1-macro': round(gnb_f1, 4),
        'Час навчання (с)': round(gnb_fit_time, 6),
        'Час передбачення (с)': round(gnb_predict_time, 6)
    },
    {
        'Модель / Метод': 'MultinomialNB (alpha=1.0) [Text]',
        'Тип навчання': 'Eager (Жадібне)',
        'Accuracy': round(mnb_results[1.0]['accuracy'], 4),
        'F1-macro': round(mnb_results[1.0]['f1_macro'], 4),
        'Час навчання (с)': round(mnb_results[1.0]['fit_time'], 6),
        'Час передбачення (с)': round(mnb_results[1.0]['predict_time'], 6)
    }
])

comparison_df

,Модель / Метод,Тип навчання,Accuracy,F1-macro,Час навчання (с),Час передбачення (с)
0,k-NN (k=21) [DocVerify],Lazy (Ліниве),0.3962,0.4084,0.011666,0.016718
1,GaussianNB [DocVerify],Eager (Жадібне),0.3297,0.3053,0.003979,0.001834
2,MultinomialNB (alpha=1.0) [Text],Eager (Жадібне),0.9454,0.9387,0.007852,0.001748


## 5. Підсумкові висновки

1. **Оцінка GaussianNB на даних DocVerify:** Classical GaussianNB показав високу швидкість виконання, однак поступається $k$-NN за точності ($F_1$-score). Це обумовлено сильним припущенням про умовну незалежність ознак та нормальний розподіл, яке порушується на практичних геометричних та яскравісних параметрах документів.
2. **Вплив згладжування Лапласа ($\alpha$):** Експерименти з `MultinomialNB` довели критичну важливість параметра $\alpha$. При $\alpha=0.0$ модель страждає від проблеми нульової ймовірності (якщо слово відсутнє в навчальному класі, добуток обнуляється). Встановлення $\alpha=1.0$ стабілізує оцінки ймовірностей.
3. **Lazy vs Eager Learning:**
   - **$k$-NN є лінивим (Lazy) класифікатором:** етап `fit` майже миттєвий, оскільки алгоритм лише зберігає дані. Проте фаза `predict` вимагає значних ресурсів через обчислення відстаней до всіх точок.
   - **Naive Bayes є жадібним (Eager) класифікатором:** під час `fit` обчислюється підсумкова статистика (апріорні ймовірності та параметри розподілів), завдяки чому фаза `predict` відбувається практично миттєво.